<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/options_flow_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import json
from datetime import datetime, date
import warnings
warnings.filterwarnings("ignore")
# ensure reproducibility
random.seed(42)
print("Libraries Installed!")

Libraries Installed!


## Configuration

In [3]:

CSV_PATH = "options_flow_history.csv"   # change path if needed

# Score thresholds
STRONG_BUY_THRESHOLD  = 75
BUY_THRESHOLD         = 55
WATCH_THRESHOLD       = 40

# Week-over-week change thresholds for commentary
SIGNIFICANT_CHANGE    = 10   # composite score change considered meaningful
LARGE_FLOW_SHIFT      = 15   # flow score change considered large
TREND_SHIFT           = 25   # trend score change considered a trend shift

## Complimentary Engine

In [5]:

class CommentaryEngine:
    """
    Generates detailed narrative + signal for each stock
    based on current scores and week-over-week changes.
    """

    def generate(self, ticker, current, previous=None):
        """
        Parameters
        ----------
        ticker   : str
        current  : dict — this week's scores and metrics
        previous : dict or None — last week's scores and metrics

        Returns
        -------
        dict with signal, headline, and full narrative
        """

        signal    = self._signal(current, previous)
        headline  = self._headline(ticker, signal, current, previous)
        narrative = self._narrative(ticker, current, previous)

        return {
            "signal":    signal,
            "headline":  headline,
            "narrative": narrative,
        }

    # ── Signal ────────────────────────────────────────────────
    def _signal(self, current, previous):

        composite = current["composite_score"]
        trend     = current["trend_score"]
        flow      = current["flow_score"]

        # Deteriorating — override bullish scores
        if previous:
            comp_delta = composite - previous["composite_score"]
            flow_delta = flow - previous["flow_score"]

            if comp_delta < -SIGNIFICANT_CHANGE and flow_delta < -LARGE_FLOW_SHIFT:
                return "SELL / REDUCE"

            if comp_delta < -SIGNIFICANT_CHANGE:
                return "WATCH — Weakening"

        if composite >= STRONG_BUY_THRESHOLD and trend >= 75 and flow >= 60:
            return "STRONG BUY"

        if composite >= BUY_THRESHOLD and trend >= 50:
            return "BUY"

        if composite >= WATCH_THRESHOLD:
            return "WATCH"

        if trend < 25 and flow < 25:
            return "AVOID"

        return "NEUTRAL"

    # ── Headline ──────────────────────────────────────────────
    def _headline(self, ticker, signal, current, previous):

        composite = current["composite_score"]
        direction = ""

        if previous:
            delta = composite - previous["composite_score"]
            if abs(delta) >= SIGNIFICANT_CHANGE:
                direction = f" (+{delta:.1f})" if delta > 0 else f" ({delta:.1f})"

        return (
            f"{ticker} | {signal} | "
            f"Composite: {composite:.1f}{direction} | "
            f"Trend: {current['trend_score']:.0f} | "
            f"Flow: {current['flow_score']:.0f}"
        )

    # ── Full Narrative ────────────────────────────────────────
    def _narrative(self, ticker, current, previous):

        lines = []

        c_comp  = current["composite_score"]
        c_trend = current["trend_score"]
        c_flow  = current["flow_score"]
        c_bull  = current.get("bullish_ratio", 0.5)
        c_vol   = current.get("unusual_contracts", 0)
        c_prem  = current.get("total_premium", 0)
        c_pcr   = current.get("put_call_ratio", 1.0)
        c_mom   = current.get("momentum_3m", 0)

        # ── Trend commentary
        if c_trend >= 75:
            lines.append(
                f"Trend structure is strong — price is above both the 10W and 30W "
                f"SMAs with the longer average rising, confirming a healthy uptrend."
            )
        elif c_trend >= 50:
            lines.append(
                f"Trend is constructive but not fully aligned. "
                f"Price is holding above key moving averages but momentum needs confirmation."
            )
        elif c_trend >= 25:
            lines.append(
                f"Trend structure is mixed. Price may be below one or more key moving averages. "
                f"Risk of further weakness if support breaks."
            )
        else:
            lines.append(
                f"Trend is broken. Price is below key moving averages and momentum "
                f"is negative — avoid new long exposure."
            )

        # ── 3M momentum
        if c_mom > 0.10:
            lines.append(
                f"3-month price momentum is strong at +{c_mom*100:.1f}%, "
                f"indicating sustained buying pressure."
            )
        elif c_mom > 0:
            lines.append(
                f"3-month momentum is modestly positive at +{c_mom*100:.1f}%."
            )
        elif c_mom < -0.10:
            lines.append(
                f"3-month momentum is deeply negative at {c_mom*100:.1f}% — "
                f"institutional sellers may be in control."
            )
        else:
            lines.append(
                f"3-month momentum is slightly negative at {c_mom*100:.1f}%."
            )

        # ── Options flow commentary
        if c_bull >= 0.70:
            lines.append(
                f"Options flow is decisively bullish — call premium represents "
                f"{c_bull*100:.0f}% of total premium, indicating institutional "
                f"positioning to the upside."
            )
        elif c_bull >= 0.55:
            lines.append(
                f"Options flow is leaning bullish with calls making up "
                f"{c_bull*100:.0f}% of total premium. Moderate institutional interest."
            )
        elif c_bull <= 0.35:
            lines.append(
                f"Options flow is bearish — put premium dominates at "
                f"{(1-c_bull)*100:.0f}% of total. Smart money may be hedging or shorting."
            )
        else:
            lines.append(
                f"Options flow is balanced — no clear directional bias from "
                f"institutional positioning this week."
            )

        # ── Put/Call ratio
        if c_pcr < 0.5:
            lines.append(
                f"Put/call ratio of {c_pcr:.2f} is low — market participants "
                f"are positioning aggressively for upside."
            )
        elif c_pcr > 1.5:
            lines.append(
                f"Put/call ratio of {c_pcr:.2f} is elevated — hedging or "
                f"outright bearish bets are rising."
            )

        # ── Unusual activity
        if c_vol > 20:
            lines.append(
                f"Unusual options activity is high with {c_vol} contracts "
                f"showing volume significantly above open interest — a strong "
                f"signal of informed institutional positioning."
            )
        elif c_vol > 10:
            lines.append(
                f"{c_vol} contracts showed unusual volume vs open interest — "
                f"moderate institutional footprint detected."
            )
        else:
            lines.append(
                f"Unusual options activity is limited this week — "
                f"flow is largely routine."
            )

        # ── Premium size
        if c_prem >= 10_000_000:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is significant — "
                f"this is institutional-scale activity."
            )
        elif c_prem >= 5_000_000:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is meaningful "
                f"but not at extreme levels."
            )
        elif c_prem > 0:
            lines.append(
                f"Total options premium of ${c_prem/1e6:.1f}M is modest — "
                f"retail-scale activity this week."
            )

        # ── Week-over-week changes
        if previous:
            comp_delta  = c_comp  - previous["composite_score"]
            flow_delta  = c_flow  - previous["flow_score"]
            trend_delta = c_trend - previous["trend_score"]

            wow_lines = []

            if abs(comp_delta) >= SIGNIFICANT_CHANGE:
                direction = "improved" if comp_delta > 0 else "deteriorated"
                wow_lines.append(
                    f"Composite score has {direction} by {abs(comp_delta):.1f} "
                    f"points week-over-week "
                    f"({previous['composite_score']:.1f} → {c_comp:.1f})."
                )

            if abs(flow_delta) >= LARGE_FLOW_SHIFT:
                direction = "surged" if flow_delta > 0 else "dropped"
                wow_lines.append(
                    f"Options flow score {direction} by {abs(flow_delta):.1f} points "
                    f"— a significant shift in institutional positioning."
                )
            elif abs(flow_delta) >= 5:
                direction = "ticked up" if flow_delta > 0 else "eased"
                wow_lines.append(
                    f"Flow score {direction} modestly by {abs(flow_delta):.1f} points."
                )

            if abs(trend_delta) >= TREND_SHIFT:
                direction = "strengthened" if trend_delta > 0 else "weakened"
                wow_lines.append(
                    f"Trend structure {direction} materially "
                    f"({previous['trend_score']:.0f} → {c_trend:.0f}) — "
                    f"{'bullish development' if trend_delta > 0 else 'watch for further deterioration'}."
                )

            if wow_lines:
                lines.append("\nWEEK-OVER-WEEK:")
                lines.extend(wow_lines)
            else:
                lines.append(
                    "\nWEEK-OVER-WEEK: Scores are largely stable — "
                    "no significant shift in trend or flow this week."
                )

        else:
            lines.append(
                "\nWEEK-OVER-WEEK: No prior week data available — "
                "this is the first recorded run for this ticker."
            )

        return " ".join(lines)

## HISTORY MANAGER — CSV persistence

In [6]:

class FlowHistoryManager:
    """
    Saves and loads week-over-week flow data from CSV.
    """

    def __init__(self, path=CSV_PATH):
        self.path = path

    def load(self):
        if os.path.exists(self.path):
            return pd.read_csv(self.path, parse_dates=["run_date"])
        return pd.DataFrame()

    def save(self, records: list):
        new_df = pd.DataFrame(records)

        if os.path.exists(self.path):
            existing = pd.read_csv(self.path, parse_dates=["run_date"])
            combined = pd.concat([existing, new_df], ignore_index=True)
            # deduplicate — keep latest run per ticker per week
            combined["week"] = pd.to_datetime(
                combined["run_date"]
            ).dt.to_period("W")
            combined = combined.drop_duplicates(
                subset=["ticker", "week"], keep="last"
            ).drop(columns=["week"])
        else:
            combined = new_df

        combined.to_csv(self.path, index=False)
        print(f"\nHistory saved to: {self.path}")

    def get_previous(self, ticker, current_run_date):
        """Get the most recent prior week record for a ticker."""
        df = self.load()
        if df.empty:
            return None

        ticker_history = df[
            (df["ticker"] == ticker)
            & (pd.to_datetime(df["run_date"]) < pd.to_datetime(current_run_date))
        ].sort_values("run_date", ascending=False)

        if ticker_history.empty:
            return None

        return ticker_history.iloc[0].to_dict()

## Main Engine

In [20]:

class OptionsFlowEngine:
    """
    Institutional Options Flow Engine with:
    - Improved flow scoring (OI-weighted, premium-adjusted)
    - Improved trend scoring (slope-aware)
    - Week-over-week tracking via CSV
    - Detailed narrative commentary with BUY/SELL/WATCH signal
    - Full ranked output with history
    """

    def __init__(self, tickers: list, csv_path: str = CSV_PATH):
        self.tickers      = tickers
        self.price_data   = {}
        self.options_data = {}
        self.flow_scores  = []
        self.history      = FlowHistoryManager(csv_path)
        self.commentary   = CommentaryEngine()
        self.run_date     = date.today().isoformat()

    # =========================================================
    # 1. PRICE DATA
    # =========================================================
    def get_price_data(self):
        print("Fetching price data...")
        for ticker in self.tickers:
            try:
                stock = yf.Ticker(ticker)
                hist  = stock.history(period="1y")   # 1 year for slope analysis
                if isinstance(hist.columns, pd.MultiIndex):
                    hist.columns = hist.columns.get_level_values(0)
                self.price_data[ticker] = hist
            except Exception as e:
                print(f"  Price error {ticker}: {e}")

    # =========================================================
    # 5. Run
    # =========================================================
    def run(self):

        self.get_price_data()
        self.get_options_data()

        records  = []
        rankings = []

        print(f"\nAnalyzing {len(self.tickers)} tickers...\n")
        print("=" * 70)

        for ticker in self.tickers:

            try:

                if ticker not in self.price_data:
                    continue
                if ticker not in self.options_data:
                    print(f"{ticker}: No options data — skipping")
                    continue

                trend_score          = self.calculate_trend_score(ticker)
                flow_score, metrics  = self.calculate_flow_score(ticker)
                composite            = round(
                    (trend_score * 0.45) + (flow_score * 0.55), 2
                )

                # 3M momentum for commentary
                close   = self.price_data[ticker]["Close"]
                mom_3m  = float((close.iloc[-1] / close.iloc[-63]) - 1) \
                          if len(close) >= 63 else 0.0

                current = {
                    "composite_score":  composite,
                    "trend_score":      trend_score,
                    "flow_score":       flow_score,
                    "momentum_3m":      round(mom_3m, 4),
                    **metrics,
                }

                # load previous week
                previous = self.history.get_previous(ticker, self.run_date)

                # generate commentary
                commentary = self.commentary.generate(ticker, current, previous)

                # print to console
                print(f"\n{'─'*70}")
                print(f"  {commentary['headline']}")
                print(f"{'─'*70}")
                print(f"  {commentary['narrative']}\n")

                # record for CSV
                records.append({
                    "run_date":          self.run_date,
                    "ticker":            ticker,
                    "composite_score":   composite,
                    "trend_score":       trend_score,
                    "flow_score":        flow_score,
                    "momentum_3m":       round(mom_3m * 100, 2),
                    "bullish_ratio":     metrics.get("bullish_ratio"),
                    "put_call_ratio":    metrics.get("put_call_ratio"),
                    "unusual_contracts": metrics.get("unusual_contracts"),
                    "total_premium":     metrics.get("total_premium"),
                    "signal":            commentary["signal"],
                    "narrative":         commentary["narrative"],
                })

                rankings.append({
                    "ticker":           ticker,
                    "signal":           commentary["signal"],
                    "composite_score":  composite,
                    "trend_score":      trend_score,
                    "flow_score":       flow_score,
                    "momentum_3m_pct":  round(mom_3m * 100, 2),
                    "bullish_ratio":    metrics.get("bullish_ratio"),
                    "unusual_count":    metrics.get("unusual_contracts"),
                    "total_premium_m":  round(
                                            metrics.get("total_premium", 0) / 1e6, 2
                                        ),
                })

            except Exception as e:
                print(f"{ticker}: ERROR — {e}")
                continue

        # save to CSV
        if records:
            self.history.save(records)

        # sort and store
        self.flow_scores = sorted(
            rankings, key=lambda x: x["composite_score"], reverse=True
        )

        return self.flow_scores




    # =========================================================
    # 2. OPTIONS DATA
    # =========================================================
    def get_options_data(self):
        print("Fetching options data...")
        for ticker in self.tickers:
            try:
                stock   = yf.Ticker(ticker)
                expiries = stock.options

                if not expiries:
                    continue

                all_options = []

                for expiry in expiries[:4]:    # 4 expiries for better coverage
                    chain = stock.option_chain(expiry)

                    calls          = chain.calls.copy()
                    puts           = chain.puts.copy()
                    calls["type"]  = "CALL"
                    puts["type"]   = "PUT"
                    calls["expiry"] = expiry
                    puts["expiry"]  = expiry

                    # days to expiration
                    exp_date = pd.to_datetime(expiry)
                    today    = pd.Timestamp.today()
                    dte      = (exp_date - today).days

                    calls["daysToExpiration"] = dte
                    puts["daysToExpiration"]  = dte

                    all_options.append(pd.concat([calls, puts]))

                if all_options:
                    self.options_data[ticker] = pd.concat(
                        all_options, ignore_index=True
                    )

            except Exception as e:
                print(f"  Options error {ticker}: {e}")

    # =========================================================
    # 3. TREND SCORE — slope-aware
    # =========================================================
    def calculate_trend_score(self, ticker):

        df    = self.price_data[ticker].copy()
        close = df["Close"]

        sma10w = close.rolling(50).mean()    # ~10 weeks on daily
        sma30w = close.rolling(150).mean()   # ~30 weeks on daily

        latest = close.iloc[-1]
        score  = 0

        # Price vs SMAs
        if latest > sma10w.iloc[-1]:
            score += 20
        if latest > sma30w.iloc[-1]:
            score += 20

        # 30W SMA slope — rising = healthy trend
        sma30_slope = sma30w.iloc[-1] - sma30w.iloc[-10]
        if sma30_slope > 0:
            score += 20

        # 10W SMA slope
        sma10_slope = sma10w.iloc[-1] - sma10w.iloc[-5]
        if sma10_slope > 0:
            score += 15

        # 3-month momentum
        if len(close) >= 63:
            mom = (latest / close.iloc[-63]) - 1
            if mom > 0.10:
                score += 25
            elif mom > 0:
                score += 15
            elif mom < -0.10:
                score -= 15

        return min(max(score, 0), 100)

    # =========================================================
    # 6. TOP STOCKS SUMMARY
    # =========================================================
    def top_stocks(self, n=10):

        print(f"\n{'='*70}")
        print(f"  TOP {n} RANKED STOCKS")
        print(f"{'='*70}")
        print(
            f"{'#':<4}{'Ticker':<8}{'Signal':<22}"
            f"{'Composite':>10}{'Trend':>8}{'Flow':>8}"
            f"{'Mom%':>8}{'Bull%':>8}{'Prem$M':>9}"
        )
        print("─" * 85)

        for i, s in enumerate(self.flow_scores[:n], 1):
            bull_pct = round((s.get("bullish_ratio") or 0) * 100, 1)
            print(
                f"{i:<4}{s['ticker']:<8}{s['signal']:<22}"
                f"{s['composite_score']:>10.1f}"
                f"{s['trend_score']:>8.0f}"
                f"{s['flow_score']:>8.0f}"
                f"{s['momentum_3m_pct']:>8.1f}"
                f"{bull_pct:>8.1f}"
                f"{s['total_premium_m']:>9.2f}"
            )

    # =========================================================
    # 7.   WEEKLY HISTORY REPORT
    # =========================================================
    def history_report(self, ticker: str):
        """Print full week-over-week history for a single ticker."""

        df = self.history.load()

        if df.empty:
            print("No history available yet.")
            return

        ticker_df = df[df["ticker"] == ticker].sort_values(
            "run_date", ascending=False
        )

        if ticker_df.empty:
            print(f"No history found for {ticker}.")
            return

        print(f"\n{'='*70}")
        print(f"  HISTORY REPORT: {ticker}")
        print(f"{'='*70}")
        print(
            f"{'Date':<14}{'Signal':<22}{'Composite':>10}"
            f"{'Trend':>8}{'Flow':>8}{'Mom%':>8}"
        )
        print("─" * 72)

        for _, row in ticker_df.iterrows():
            print(
                f"{str(row['run_date']):<14}"
                f"{str(row.get('signal','')):<22}"
                f"{row['composite_score']:>10.1f}"
                f"{row['trend_score']:>8.0f}"
                f"{row['flow_score']:>8.0f}"
                f"{row.get('momentum_3m', 0):>8.1f}"
            )

    # =========================================================
    # 4. OPTIONS FLOW SCORE — improved
    # =========================================================
    def calculate_flow_score(self, ticker):

        df    = self.options_data[ticker].copy()
        score = 0

        # clean volume and OI
        df["volume"]       = pd.to_numeric(df["volume"],       errors="coerce").fillna(0)
        df["openInterest"] = pd.to_numeric(df["openInterest"], errors="coerce").fillna(0)
        df["lastPrice"]    = pd.to_numeric(df["lastPrice"],    errors="coerce").fillna(0)

        # ── Premium
        df["premium"]  = df["lastPrice"] * df["volume"] * 100

        call_df        = df[df["type"] == "CALL"]
        put_df         = df[df["type"] == "PUT"]

        call_premium   = call_df["premium"].sum()
        put_premium    = put_df["premium"].sum()
        total_premium  = call_premium + put_premium

        if total_premium == 0:
            return 0, {}

        bullish_ratio  = call_premium / total_premium

        # ── Put/Call ratio by volume
        call_volume    = call_df["volume"].sum()
        put_volume     = put_df["volume"].sum()
        put_call_ratio = put_volume / (call_volume + 1)

        # ── Unusual activity — vol significantly above OI
        df["vol_oi_ratio"] = df["volume"] / (df["openInterest"] + 1)
        unusual            = df[df["vol_oi_ratio"] > 2]
        unusual_count      = len(unusual)

        # ── Scoring
        # Bullish premium positioning
        if bullish_ratio >= 0.70:
            score += 30
        elif bullish_ratio >= 0.55:
            score += 15
        elif bullish_ratio <= 0.35:
            score -= 15

        # Call volume dominance
        if call_volume > put_volume * 1.5:
            score += 20
        elif put_volume > call_volume * 1.5:
            score -= 10

        # Unusual contract count
        if unusual_count > 20:
            score += 25
        elif unusual_count > 10:
            score += 15
        elif unusual_count > 5:
            score += 8

        # Total premium size — institutional scale
        if total_premium >= 10_000_000:
            score += 20
        elif total_premium >= 5_000_000:
            score += 12
        elif total_premium >= 1_000_000:
            score += 6

        # Short-dated dominance penalty — retail noise
        short_dated = df[df["daysToExpiration"] < 7]
        if len(df) > 0 and len(short_dated) / len(df) > 0.5:
            score -= 15

        # Put/call ratio confirmation
        if put_call_ratio < 0.5:
            score += 10
        elif put_call_ratio > 1.5:
            score -= 10

        metrics = {
            "bullish_ratio":      round(bullish_ratio, 3),
            "put_call_ratio":     round(put_call_ratio, 3),
            "unusual_contracts":  unusual_count,
            "total_premium":      round(total_premium, 2),
            "call_premium":       round(call_premium, 2),
            "put_premium":        round(put_premium, 2),
        }

        return min(max(score, 0), 100), metrics

In [23]:
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    tickers = [
        "NVDA", "META", "AVGO", "MSFT", "PLTR", "GLW",
        "AAPL", "AMZN", "TSLA", "JPM", "V","WDC", "BE"
    ]

    engine = OptionsFlowEngine(tickers)

    # Run full analysis — saves to CSV automatically
    rankings = engine.run()

    # Print ranked table
    engine.top_stocks(n=9)

    # Print full history for a specific ticker
    engine.history_report("NVDA")


Fetching price data...
Fetching options data...

Analyzing 13 tickers...

NVDA: ERROR — time data "2026-05-16" doesn't match format "%Y-%m-%d %H:%M:%S", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
META: ERROR — time data "2026-05-16" doesn't match format "%Y-%m-%d %H:%M:%S", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
AVGO: ERROR — time data "2026-05-16" doesn't mat